# Video Indexing at Speed - Kaggle Execution Notebook

### Instructions:
1. **Settings -> Accelerator**: Set to **GPU T4 x 2** (Dual T4 for 2-GPU sharding) or **GPU P100**.
2. **Settings -> Internet**: Toggle **ON**.
3. **HuggingFace Access**: Accept dataset terms at [huggingface.co/datasets/HuggingFaceFV/finevideo](https://huggingface.co/datasets/HuggingFaceFV/finevideo) and create a Read Token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [1]:
import glob, os, shutil

# 1. Clean working directory of old code
%cd /kaggle/working

# 2. Check if Kaggle uploaded as a zip or auto-extracted folder
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
if zip_files:
    print(f"Found zip: {zip_files[0]}, unzipping...")
    !unzip -q -o "{zip_files[0]}" -d /kaggle/working/
else:
    print("Found extracted dataset, copying files...")
    run_paths = glob.glob("/kaggle/input/**/run.py", recursive=True)
    if not run_paths:
        raise FileNotFoundError("Could not find run.py in /kaggle/input! Make sure your dataset is attached.")
    src_dir = os.path.dirname(run_paths[0])
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join("/kaggle/working", item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)

# 3. Verify files and check that the ndim fix is in detector.py
print("\n--- Verifying detector.py has the fix ---")
!grep -n "ndim" /kaggle/working/vidindex/detector.py && echo "Fix confirmed!" || echo "WARNING: Fix not found!"

/kaggle/working
Found extracted dataset, copying files...

--- Verifying detector.py has the fix ---
74:        if hasattr(batch_images, "ndim") and batch_images.ndim == 4:
Fix confirmed!


In [2]:
!pip install -q ultralytics pyarrow av opencv-python-headless huggingface_hub tqdm

In [3]:
# Step 2: Install Python Requirements
!pip install --upgrade pip
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.7 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8

In [4]:
# Step 3: Run Pure-Python Unit Tests
!python -m unittest discover -s tests -v

test_all_seconds_written_including_empty (test_common.FinalizeTests.test_all_seconds_written_including_empty) ... ok
test_error_keeps_earlier_seconds_and_adds_error_line (test_common.FinalizeTests.test_error_keeps_earlier_seconds_and_adds_error_line) ... ok
test_empty (test_common.MedianTests.test_empty) ... ok
test_even_rounds_down (test_common.MedianTests.test_even_rounds_down) ... ok
test_odd (test_common.MedianTests.test_odd) ... ok
test_mask_bits_follow_object_order (test_common.SecondResultTests.test_mask_bits_follow_object_order) ... ok
test_no_frames (test_common.SecondResultTests.test_no_frames) ... ok
test_people_median (test_common.SecondResultTests.test_people_median) ... ok
test_present_when_at_least_half (test_common.SecondResultTests.test_present_when_at_least_half) ... ok
test_three_samples_need_two (test_common.SecondResultTests.test_three_samples_need_two) ... ok
test_bucket_of (test_decoder.DecoderMathTests.test_bucket_of) ... ok
test_display_matrix_rotation (test_de

In [5]:
# Step 4: Preparation Phase (download weights & warmup)
# Standard preparation:
!python run.py --prepare --batch-size 64

# Optional: If TensorRT is available in your environment, build FP16 engine:
# !python run.py --prepare --export-trt --batch-size 64

=== Preparation Phase ===
1. Checking Python dependencies...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
   [OK] Torch: 2.11.0+cu130 (CUDA available: True)
   [OK] GPU 0: Tesla T4
   [OK] GPU 1: Tesla T4
   [OK] Ultralytics: available
   [OK] PyAV: 19.0.2
   [OK] OpenCV: 4.14.0
   [OK] PyArrow: 23.0.1
2. Downloading / verifying official YOLO11l weights (yolo11l.pt)...
   [OK] YOLO11l loaded and warm-up inference passed.
=== Preparation Complete: Ready for timed offline execution ===


In [6]:
from huggingface_hub import login
login(token="[REMOVED]")


In [7]:
%%bash
    echo "=== /kaggle/working space (20GB limited) ==="
    df -h /kaggle/working

    echo ""
    echo "=== /tmp space (NO quota, separate partition) ==="
    df -h /tmp

    echo ""
    echo "Downloading all 50 parquet files into /tmp/finevideo/data/ ..."
    mkdir -p /tmp/finevideo/data

    hf download HuggingFaceFV/finevideo \
      --repo-type dataset \
      --include "data/train-000[0-4]*.parquet" \
      --local-dir /tmp/finevideo

    echo ""
    echo "Total files downloaded:"
    ls /tmp/finevideo/data/*.parquet | wc -l

    echo "Disk check after download:"
    df -h /tmp
    df -h /kaggle/working

=== /kaggle/working space (20GB limited) ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   68M   20G   1% /kaggle/working

=== /tmp space (NO quota, separate partition) ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  5.6T  2.5T  70% /

✓ Downloaded
  path: /tmp/finevideo

Total files downloaded:
50
Disk check after download:
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  5.6T  2.5T  70% /
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   68M   20G   1% /kaggle/working


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Fetching 50 files: 100%|██████████| 50/50 [01:37<00:00,  1.95s/it]


In [8]:
%%bash
    pkill -9 -f "run.py" || true
    rm -f /kaggle/working/shard0.log /kaggle/working/shard1.log
    rm -f /kaggle/working/results_shard0.jsonl /kaggle/working/results_shard1.jsonl

    python /kaggle/working/run.py \
      --input /tmp/finevideo/data \
      --output /kaggle/working/results_shard0.jsonl \
      --shard-index 0 --num-shards 2 \
      --device cuda:0 --workers 0 \
      --fps 5 --batch-size 64 \
      > /kaggle/working/shard0.log 2>&1 &

    python /kaggle/working/run.py \
      --input /tmp/finevideo/data \
      --output /kaggle/working/results_shard1.jsonl \
      --shard-index 1 --num-shards 2 \
      --device cuda:1 --workers 0 \
      --fps 5 --batch-size 64 \
      > /kaggle/working/shard1.log 2>&1 &

    echo "Launched. PIDs: $(pgrep -f run.py | tr '\n' ' ')"
    wait
    echo "=== Both done! ==="

Process is interrupted.


In [9]:
%%bash
    echo "=== 1. Check ALL .jsonl files in /kaggle/working ==="
    ls -lh /kaggle/working/*.jsonl 2>/dev/null || echo "No .jsonl files found"
    wc -l /kaggle/working/*.jsonl 2>/dev/null

    echo ""
    echo "=== 2. Check if processes are running & CPU/GPU usage ==="
    ps aux | grep run.py | grep -v grep || echo "NO run.py process running!"

    echo ""
    echo "=== 3. FULL shard0.log (last 40 lines) ==="
    tail -n 40 /kaggle/working/shard0.log 2>/dev/null || echo "No shard0.log"

    echo ""
    echo "=== 4. FULL shard1.log (last 40 lines) ==="
    tail -n 40 /kaggle/working/shard1.log 2>/dev/null || echo "No shard1.log"

=== 1. Check ALL .jsonl files in /kaggle/working ===
-rw-r--r-- 1 root root   0 Nov 21 20:20 /kaggle/working/results.shard0of2.jsonl
-rw-r--r-- 1 root root 17M Nov 21 22:00 /kaggle/working/results_shard0.shard0of2.jsonl
-rw-r--r-- 1 root root   0 Nov 21 20:20 /kaggle/working/results.shard1of2.jsonl
-rw-r--r-- 1 root root 17M Nov 21 22:00 /kaggle/working/results_shard1.shard1of2.jsonl
       0 /kaggle/working/results.shard0of2.jsonl
  189092 /kaggle/working/results_shard0.shard0of2.jsonl
       0 /kaggle/working/results.shard1of2.jsonl
  182353 /kaggle/working/results_shard1.shard1of2.jsonl
  371445 total

=== 2. Check if processes are running & CPU/GPU usage ===
root         501  125 12.8 26415696 4222824 ?    Rl   20:23 122:28 python3 /kaggle/working/run.py --input /tmp/finevideo/data --output /kaggle/working/results_shard0.jsonl --shard-index 0 --num-shards 2 --device cuda:0 --workers 0 --fps 5 --batch-size 64
root         502  123 10.4 24385884 3448312 ?    Rl   20:23 120:40 python3

In [16]:
from IPython.display import FileLink, display

print("=== 1. Checking all shard files ===")
!ls -lh /kaggle/working/*.jsonl
!wc -l /kaggle/working/*.jsonl

print("\n=== 2. Merging results_shard0 & results_shard1 ===")
!rm -f /kaggle/working/results.jsonl
!cat /kaggle/working/results_shard0.shard0of2.jsonl /kaggle/working/results_shard1.shard1of2.jsonl > /kaggle/working/results.jsonl
!wc -l /kaggle/working/results.jsonl

print("\n=== 3. Compressing to results.jsonl.gz ===")
!gzip -k -9 -f /kaggle/working/results.jsonl
!ls -lh /kaggle/working/results.jsonl.gz

print("\n=== 4. Download Link ===")
display(FileLink('results.jsonl.gz'))

=== 1. Checking all shard files ===
-rw-r--r-- 1 root root   0 Nov 21 20:20 /kaggle/working/results.shard0of2.jsonl
-rw-r--r-- 1 root root 18M Nov 21 22:04 /kaggle/working/results_shard0.shard0of2.jsonl
-rw-r--r-- 1 root root   0 Nov 21 20:20 /kaggle/working/results.shard1of2.jsonl
-rw-r--r-- 1 root root 17M Nov 21 22:05 /kaggle/working/results_shard1.shard1of2.jsonl
       0 /kaggle/working/results.shard0of2.jsonl
  193541 /kaggle/working/results_shard0.shard0of2.jsonl
       0 /kaggle/working/results.shard1of2.jsonl
  186909 /kaggle/working/results_shard1.shard1of2.jsonl
  380450 total

=== 2. Merging results_shard0 & results_shard1 ===
380450 /kaggle/working/results.jsonl

=== 3. Compressing to results.jsonl.gz ===
-rw-r--r-- 1 root root 1.1M Nov 21 22:05 /kaggle/working/results.jsonl.gz

=== 4. Download Link ===


/kaggle/working/results.jsonl.gz